# 🔬 Surgical Organ Classifier — PaliGemma 2 Fine-Tuning

Fine-tunes **PaliGemma 2** (`google/paligemma2-3b-pt-224`) for surgical organ
classification using laparoscopic images.  Supports multi-dataset training
with configurable per-dataset weights so you can prioritise domain-specific
data (e.g. DSAD for colorectal procedures).

**Recommended runtime:** GPU (T4 or A100)  
**Runtime → Change runtime type → GPU**

---

## Quick-start

1. Add `HF_TOKEN` to the Secrets panel (🔑) — needed to load PaliGemma 2.
2. Add `KAGGLE_USERNAME` + `KAGGLE_KEY` to Secrets — needed to download DSAD.
   Get your key at: **https://www.kaggle.com/settings/account** → API → Create New Token.
3. Run **Section 1** to install dependencies.
4. Run **Section 2** to clone your repo (or upload files manually).
5. Run **Section 3** to set up credentials and download DSAD.
6. Run **Section 4** to configure weights and train.
7. Run **Section 5** to evaluate and export.

---

## Dataset weight guide

| Dataset | Weight | Rationale |
|---------|--------|-----------|
| `dsad` | `3.0` | Colorectal-specific, expert pixel labels |
| `surgeon_corrections` | `5.0` | Your ground-truth annotations |
| `cholec80` | `1.0` | General laparoscopic, lower priority |
| `surgisr4k` | `0.0` | No organ labels — skip |

---

## DSAD source

DSAD is **not** on HuggingFace. It is hosted on:
- **Kaggle** (automated): `anindyamajumder/the-dresden-surgical-anatomy-dataset`
- **Figshare** (manual): https://springernature.figshare.com/articles/dataset/The_Dresden_Surgical_Anatomy_Dataset_for_abdominal_organ_segmentation_in_surgical_data_science/21702600

---
## 1 · Install dependencies

In [ ]:
%%capture
!pip install -q \
    torch torchvision \
    transformers>=4.40.0 \
    peft \
    accelerate \
    datasets \
    Pillow \
    numpy \
    pycocotools \
    scikit-learn \
    matplotlib \
    tqdm \
    huggingface_hub \
    opencv-python-headless \
    kaggle

print('✓ Dependencies installed')

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ No GPU detected. Switch runtime to GPU for reasonable training speed.')

---
## 2 · Load training code

Option A — clone from GitHub (recommended).  
Option B — upload `finetune.py`, `dataset.py`, `download_public_data.py` manually.

In [ ]:
import os

# ── Configure ──────────────────────────────────────────────────────────
REPO_URL    = 'https://github.com/YOUR_ORG/surgical-annotator.git'  # ← edit
REPO_BRANCH = 'main'
REPO_DIR    = '/content/surgical-annotator'
# ───────────────────────────────────────────────────────────────────────

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest...')
    !git -C {REPO_DIR} pull
else:
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}

# Add training/ to Python path so import works without package install
import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'training'))
print('✓ Training code available')

---
## 3 · Set up credentials & download DSAD

DSAD is hosted on **Kaggle**, not HuggingFace.  
The cell below reads `KAGGLE_USERNAME` and `KAGGLE_KEY` from Colab Secrets (🔑).  
To get your key: https://www.kaggle.com/settings/account → API → **Create New Token**.

### 3A · Configure credentials

In [ ]:
# ── Configure ──────────────────────────────────────────────────────────
DSAD_MAX_SAMPLES = 2000   # Full dataset = 13 195; reduce for a quick test
DATASETS_ROOT    = '/content/datasets'
# ───────────────────────────────────────────────────────────────────────

import os
from google.colab import userdata

# ── Kaggle credentials (for DSAD download) ────────────────────────────
# DSAD lives on Kaggle: anindyamajumder/the-dresden-surgical-anatomy-dataset
# Steps to get your API key:
#   1. Visit https://www.kaggle.com/settings/account
#   2. API → Create New Token  (downloads kaggle.json)
#   3. Add KAGGLE_USERNAME and KAGGLE_KEY to the Colab Secrets panel (🔑)
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    print('✓ Kaggle credentials loaded from Secrets')
except Exception as e:
    print(f'⚠ Kaggle credentials not found in Secrets: {e}')
    print()
    print('Option A — add KAGGLE_USERNAME + KAGGLE_KEY to Colab Secrets (🔑)')
    print('Option B — upload kaggle.json manually:')
    print('  from google.colab import files; files.upload()  # select kaggle.json')
    print('  !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json')

# ── HuggingFace credentials (for PaliGemma 2 model weights) ──────────
try:
    import huggingface_hub
    huggingface_hub.login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('✓ HuggingFace credentials loaded')
except Exception as e:
    print(f'⚠ HuggingFace login skipped: {e}')
    print('  Add HF_TOKEN to Colab Secrets (🔑) if PaliGemma 2 download fails.')

### 3B · Download DSAD from Kaggle

In [ ]:
from download_public_data import download_dsad

dsad_coco = download_dsad(
    output_dir=DATASETS_ROOT,
    max_samples=DSAD_MAX_SAMPLES,
)
DSAD_IMAGES_DIR = f'{DATASETS_ROOT}/dsad/images'
print(f'\nDSAD COCO JSON : {dsad_coco}')
print(f'DSAD images    : {DSAD_IMAGES_DIR}')

### 3C · Manual Figshare alternative

If Kaggle auth is unavailable, download DSAD manually from Figshare (free, no registration):

1. Go to: https://springernature.figshare.com/articles/dataset/The_Dresden_Surgical_Anatomy_Dataset_for_abdominal_organ_segmentation_in_surgical_data_science/21702600
2. Click **Download all** → save the ZIP.
3. Upload here and run the cell below.

In [ ]:
# ── Only run this if Kaggle download failed ───────────────────────────
# Upload the DSAD zip downloaded from Figshare, then run this cell.

from google.colab import files
import shutil, os

dsad_dir = f'{DATASETS_ROOT}/dsad'
os.makedirs(dsad_dir, exist_ok=True)

print('Select the DSAD zip file downloaded from Figshare:')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = os.path.join(dsad_dir, fname)
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'Uploaded {fname} → {dest}')
    print('The downloader will auto-unzip on next run.')

# Re-run the downloader — it will find the zip and process it
from download_public_data import download_dsad
dsad_coco       = download_dsad(DATASETS_ROOT, max_samples=DSAD_MAX_SAMPLES)
DSAD_IMAGES_DIR = f'{DATASETS_ROOT}/dsad/images'
print(f'\nDSAD COCO JSON : {dsad_coco}')

### 3D · Upload your own surgeon-corrected annotations (optional)

Export a COCO JSON from the annotation app, then upload it here.

In [ ]:
from google.colab import files
import shutil, os

OWN_ANNOTATIONS_DIR = '/content/own_annotations'
OWN_IMAGES_DIR      = '/content/own_images'
os.makedirs(OWN_ANNOTATIONS_DIR, exist_ok=True)
os.makedirs(OWN_IMAGES_DIR,      exist_ok=True)

print('Upload your COCO JSON export(s) and image archive (.zip) below.')
print('(Skip this cell if you have no surgeon-corrected data yet.)')

uploaded = files.upload()
for fname, data in uploaded.items():
    dest = os.path.join(OWN_ANNOTATIONS_DIR, fname)
    with open(dest, 'wb') as f:
        f.write(data)
    if fname.endswith('.zip'):
        shutil.unpack_archive(dest, OWN_IMAGES_DIR)
        print(f'  Extracted {fname} → {OWN_IMAGES_DIR}')
    else:
        print(f'  Saved {fname} → {dest}')

### 3E · (Optional) Download other public datasets

In [ ]:
# Uncomment datasets you want:

# from download_public_data import download_cholec80
# cholec_coco = download_cholec80(DATASETS_ROOT, max_samples=1000)

# from download_public_data import download_roboflow_surgical
# roboflow_coco = download_roboflow_surgical(DATASETS_ROOT, max_samples=500)

print('Uncomment the lines above to download additional datasets.')

---
## 4 · Configure training

In [ ]:
import os

# ── Dataset specs: path/to/coco.json:path/to/images:weight ───────────
# Adjust paths and weights for your setup.
# Remove entries for datasets you have not downloaded.
DATASET_SPECS = [
    # DSAD — colorectal-specific, expert labels → high weight
    f'{DATASETS_ROOT}/dsad/annotations.json:{DATASETS_ROOT}/dsad/images:3.0',

    # Surgeon corrections — your ground truth → highest weight
    # f'{OWN_ANNOTATIONS_DIR}/export.json:{OWN_IMAGES_DIR}:5.0',

    # Cholec80 — general laparoscopic → lower weight
    # f'{DATASETS_ROOT}/cholec80/annotations.json:{DATASETS_ROOT}/cholec80/images:1.0',
]

# ── Model ─────────────────────────────────────────────────────────────
MODEL_ID     = 'google/paligemma2-3b-pt-224'
OUTPUT_DIR   = '/content/checkpoints'

# ── Hyperparameters ───────────────────────────────────────────────────
EPOCHS       = 10
BATCH_SIZE   = 8     # Reduce to 4 if OOM on T4
LR           = 2e-5
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
PATIENCE     = 3
TRAIN_RATIO  = 0.8
VAL_RATIO    = 0.1
BF16         = torch.cuda.is_available()  # auto-enable on GPU

# ── Verify dataset specs exist ────────────────────────────────────────
for spec in DATASET_SPECS:
    parts = spec.split(':')
    coco_json  = parts[0]
    images_dir = parts[1]
    weight     = float(parts[2]) if len(parts) > 2 else 1.0
    ok_json    = '✓' if os.path.exists(coco_json)  else '✗ MISSING'
    ok_imgs    = '✓' if os.path.isdir(images_dir)  else '✗ MISSING'
    print(f'  JSON {ok_json}  {coco_json}')
    print(f'  imgs {ok_imgs}  {images_dir}  (weight={weight})')
    print()

print(f'Output dir : {OUTPUT_DIR}')
print(f'BF16       : {BF16}')
print(f'Epochs     : {EPOCHS}')
print(f'Batch size : {BATCH_SIZE}')

## 4B · Run training

In [ ]:
# Build the finetune.py command
datasets_flags = ' '.join(f"'{spec}'" for spec in DATASET_SPECS)

cmd = (
    f"python {REPO_DIR}/training/finetune.py "
    f"  --datasets {' '.join(DATASET_SPECS)} "
    f"  --output_dir {OUTPUT_DIR} "
    f"  --model_id {MODEL_ID} "
    f"  --epochs {EPOCHS} "
    f"  --batch_size {BATCH_SIZE} "
    f"  --learning_rate {LR} "
    f"  --lora_r {LORA_R} "
    f"  --lora_alpha {LORA_ALPHA} "
    f"  --lora_dropout {LORA_DROPOUT} "
    f"  --patience {PATIENCE} "
    f"  --train_ratio {TRAIN_RATIO} "
    f"  --val_ratio {VAL_RATIO} "
    + ('  --bf16' if BF16 else '')
)

print('Command to run:')
print(cmd.replace('  ', '\n  '))

In [ ]:
# ⚠ This cell runs the fine-tuning — may take 30 min – 3 hours depending
# on dataset size, GPU, and number of epochs.

import subprocess, sys

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    pass

env = os.environ.copy()
if hf_token:
    env['HF_TOKEN'] = hf_token
env['PYTHONPATH'] = f"{REPO_DIR}/training:{env.get('PYTHONPATH', '')}"

proc = subprocess.Popen(
    cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print(f'\n--- finetune.py exited with code {proc.returncode} ---')
if proc.returncode != 0:
    print('Training FAILED. Check the output above for errors.')
else:
    print('✓ Training complete!')

---
## 5 · Evaluate

In [ ]:
EVAL_DATASET_SPEC = DATASET_SPECS[0]  # evaluate on first dataset by default
parts    = EVAL_DATASET_SPEC.split(':')
EVAL_JSON    = parts[0]
EVAL_IMAGES  = parts[1]
CHECKPOINT   = f'{OUTPUT_DIR}/best_model'
EVAL_OUT_DIR = '/content/evaluation_results'

eval_cmd = (
    f"python {REPO_DIR}/training/evaluate.py "
    f"  --checkpoint {CHECKPOINT} "
    f"  --coco_exports {EVAL_JSON} "
    f"  --images_dir {EVAL_IMAGES} "
    f"  --output_dir {EVAL_OUT_DIR}"
)

print('Running evaluation...')
!{eval_cmd}

# Show confusion matrix if it was generated
cm_path = f'{EVAL_OUT_DIR}/confusion_matrix.png'
if os.path.exists(cm_path):
    from IPython.display import Image as IPyImage, display
    display(IPyImage(filename=cm_path))

# Show metrics
metrics_path = f'{EVAL_OUT_DIR}/metrics.json'
if os.path.exists(metrics_path):
    import json
    with open(metrics_path) as f:
        metrics = json.load(f)
    print('\nMetrics:')
    print(json.dumps(metrics, indent=2))

---
## 6 · Export merged model for production

In [ ]:
PRODUCTION_DIR = '/content/production_model'

export_cmd = (
    f"python {REPO_DIR}/training/export_model.py "
    f"  --checkpoint {OUTPUT_DIR}/best_model "
    f"  --output {PRODUCTION_DIR}"
)

print('Exporting merged model...')
!{export_cmd}
print(f'✓ Production model saved to {PRODUCTION_DIR}')

# Show file sizes
!du -sh {PRODUCTION_DIR}/*

---
## 7 · Download or push to HuggingFace Hub

In [ ]:
# Option A: Download checkpoints as a ZIP to your local machine
import shutil
from google.colab import files

zip_path = '/content/checkpoints.zip'
shutil.make_archive('/content/checkpoints', 'zip', OUTPUT_DIR)
print(f'Created {zip_path}')
files.download(zip_path)
print('✓ Download started')

In [ ]:
# Option B: Push production model directly to HuggingFace Hub
# Fill in HF_REPO_ID before running.

HF_REPO_ID = 'your-hf-username/surgical-organ-classifier'  # ← edit

from huggingface_hub import HfApi
api = HfApi()

try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

api.upload_folder(
    folder_path=PRODUCTION_DIR,
    repo_id=HF_REPO_ID,
    repo_type='model',
    token=hf_token,
    commit_message='Fine-tuned surgical organ classifier',
)
print(f'✓ Model pushed to https://huggingface.co/{HF_REPO_ID}')

---
## 8 · Quick inference test

In [ ]:
import json
from PIL import Image
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from peft import PeftModel
import torch

CHECKPOINT = f'{OUTPUT_DIR}/best_model'
PROMPT     = 'Identify the highlighted surgical organ. Answer with the organ name only:'

print('Loading fine-tuned model...')
processor = AutoProcessor.from_pretrained(CHECKPOINT)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    CHECKPOINT,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)
model.eval()

# Load organ list written during training
organ_list_path = f'{OUTPUT_DIR}/organ_list.json'
if os.path.exists(organ_list_path):
    with open(organ_list_path) as f:
        organ_list = json.load(f)
    print(f'Organ classes: {organ_list}')

# Test with a sample image from DSAD
dsad_images = sorted([
    f for f in os.listdir(f'{DATASETS_ROOT}/dsad/images')
    if f.endswith('.jpg')
])[:5]

for fname in dsad_images:
    img_path = f'{DATASETS_ROOT}/dsad/images/{fname}'
    image    = Image.open(img_path).convert('RGB')

    inputs = processor(
        images=image,
        text=PROMPT,
        return_tensors='pt',
    ).to(model.device)

    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=8, do_sample=False)

    input_len = inputs['input_ids'].shape[1]
    pred = processor.decode(generated[0][input_len:], skip_special_tokens=True).strip()
    print(f'  {fname}  →  "{pred}"')

print('\n✓ Inference test complete')

---
## Notes

**Memory tips for T4 (15 GB VRAM)**
- If you hit OOM, set `BATCH_SIZE = 4`.
- LoRA `r=8` uses ~40% less memory than the default `r=16`.

**Enable BF16**  
Automatically enabled if a GPU is detected.  Halves VRAM usage and speeds up training ~2×.

**Resuming from a checkpoint**  
Set `CHECKPOINT` to point to `checkpoints/latest` and add `--resume_from_checkpoint` support in `finetune.py` (coming soon).

**Deploying the model**  
After export, copy `production_model/` to your server and set:
```bash
export VLM_BACKEND=finetuned
export FINETUNED_MODEL_PATH=./training/production_model
```